# Music Annotation Project Notebook 64
## Exact Page/Y/X Clean-Truth Table and 59R2 Removal-Safety Audit

**Runtime:** CPU. In Colab, select **Runtime → Run all**.

### Correction to the endpoint name
For a REMOVE policy, the primary safety error is **false removal of genuine printed ink**, not false preservation. This notebook therefore compares the diagnostic 59R2 regimes at `0.995`, `0.993`, and `0.985` on:

- genuine printed-ink pixels selected for removal;
- annotation/non-print pixels selected for removal;
- truth-uncertain selected pixels;
- false-removal rate and page-clustered uncertainty;
- catastrophic page/component concentrations;
- page and source-group stability where source metadata resolves.

### Exact truth-table contract
Notebook 64 joins the Notebook 61/63 cohort to authoritative registered clean pages by exact `(page, y, x)`. It uses the frozen project printed-ink method:

```python
cv2.adaptiveThreshold(clean, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV, 41, 15)
```

Labels are:

- `PRINT`: clean printed ink exists in a one-pixel tolerance neighbourhood and the coordinate is not in the morphology-sensitive boundary band;
- `ANNOTATION_NONPRINT`: no clean printed ink exists in that neighbourhood and the coordinate is not in the boundary band;
- `UNCERTAIN`: the clean printed-ink label is boundary/tolerance sensitive.

Clean truth is loaded only after diagnostic decisions are rebuilt and hashed. It is never used to create a threshold decision.

### All-page visual output
The notebook writes one seven-panel threshold ladder for **every page**, matching the Page258 layout, plus index/contact sheets. It also writes a Page258 top-right diagnostic crop to determine whether “SCORE + 27 parts” is:

1. outside the authoritative candidate population;
2. inside the population but below each threshold;
3. blocked by page-global coherence; or
4. assigned clean PRINT / ANNOTATION_NONPRINT / UNCERTAIN truth.

Alternative thresholds remain retrospective diagnostics only.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

Mounted at /content/drive


## 1. Configuration

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, platform, shutil, sys, zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB61_ROOT = ROOT / '59r2_annotated_only_shadow_test'
NB61_RUN_OVERRIDE = None
R59R2_OVERRIDE = None
NB63_RUN_OVERRIDE = None

THRESHOLDS = [0.995, 0.993, 0.990, 0.985, 0.980, 0.975, 0.950]
SAFETY_THRESHOLDS = [0.995, 0.993, 0.985]
EXPECTED_PAGES = 151
FOCUS_PAGE = 'Page258'
TRUTH_TOLERANCE = 1
PRINT_BLOCK = 41
PRINT_C = 15
BOOTSTRAP_RESAMPLES = 10000
SEED = 42

# Optional crop for the Page258 top-right note. None means derive a generous upper-right crop.
PAGE258_TOP_RIGHT_CROP = None  # (x0, y0, x1, y1)

OUTPUT_ROOT = ROOT / 'notebook64_exact_truth_and_threshold_safety'
RUN = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook64_exact_truth_safety')
TABLES, TRUTH, AUDIT, LADDERS, CONTACTS, FOCUS, UPLOAD = [RUN/x for x in ['tables','truth','audit','threshold_ladders','contact_sheets','page258','upload']]
for p in [RUN,TABLES,TRUTH,AUDIT,LADDERS,CONTACTS,FOCUS,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Output:',RUN)

Output: /content/drive/MyDrive/MusicAnnotationProject/notebook64_exact_truth_and_threshold_safety/20261113_153058_notebook64_exact_truth_safety


## 2. Helpers and run discovery

In [3]:
def require(v,msg):
    if not v: raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def semantic_table_hash(df,cols):
    q=df[cols].sort_values(cols).reset_index(drop=True)
    return hashlib.sha256(pd.util.hash_pandas_object(q,index=False).to_numpy().tobytes()).hexdigest()
def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def image_map(folder):
    supported={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in supported} if Path(folder).exists() else {}
def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n; d=1+z*z/n; c=p+z*z/(2*n); r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c+r)/d

NB61_REQ=['tables/59r2_scored_reviewed_cohort.parquet','experiment_identity.json',f'pages/{FOCUS_PAGE}/observed.png']
NB61=Path(NB61_RUN_OVERRIDE) if NB61_RUN_OVERRIDE else latest_complete(NB61_ROOT,NB61_REQ)
require(NB61 is not None,'Complete Notebook 61 run not found. Set NB61_RUN_OVERRIDE.')
PARQUET_PATH=NB61/'tables'/'59r2_scored_reviewed_cohort.parquet'
IDENTITY61=json.loads((NB61/'experiment_identity.json').read_text())

if R59R2_OVERRIDE is not None: R59R2=Path(R59R2_OVERRIDE)
elif IDENTITY61.get('r59r2') and Path(IDENTITY61['r59r2']).exists(): R59R2=Path(IDENTITY61['r59r2'])
else: R59R2=latest_complete(ROOT,['bundle/policy.json','bundle/runtime_contract.json'])
require(R59R2 is not None,'Frozen 59R2 bundle not found.')

NB63=None
if NB63_RUN_OVERRIDE is not None: NB63=Path(NB63_RUN_OVERRIDE)
else: NB63=latest_complete(ROOT/'notebook63_59r2_diagnostic_operating_point_envelope',['tables/threshold_envelope_summary.csv','experiment_identity.json'])
print('Notebook 61:',NB61)
print('Frozen 59R2:',R59R2)
print('Notebook 63:',NB63)

Notebook 61: /content/drive/MyDrive/MusicAnnotationProject/59r2_annotated_only_shadow_test/20261112_201411_59r2_shadow
Frozen 59R2: /content/drive/MyDrive/MusicAnnotationProject/remove_challenger_59r2_component_invariant/20261112_145331_59R2_component_invariant_challenger
Notebook 63: /content/drive/MyDrive/MusicAnnotationProject/notebook63_59r2_diagnostic_operating_point_envelope/20261113_145329_notebook63_59r2_threshold_envelope


## 3. Load frozen coherence contract and compact scored cohort

In [4]:
BUNDLE=R59R2/'bundle'; policy=json.loads((BUNDLE/'policy.json').read_text()); contract=json.loads((BUNDLE/'runtime_contract.json').read_text())
def first(*xs):
    for x in xs:
        if x is not None:return x
FROZEN_THRESHOLD=float(first(contract.get('threshold'),contract.get('remove_threshold'),policy.get('threshold'),policy.get('remove_threshold')))
CONNECTIVITY=int(first(contract.get('connectivity'),policy.get('connectivity'),8))
MINIMUM_RUN=int(first(contract.get('minimum_run'),contract.get('min_run'),policy.get('minimum_run'),2))
require(abs(FROZEN_THRESHOLD-.995)<1e-12,'Frozen threshold is not 0.995.')

cols=['page','y','x','component_key','safe_remove','p_remove','p_remove_59r2','paper_value']
cohort=pd.read_parquet(PARQUET_PATH,columns=cols)
cohort['page']=cohort.page.astype(str)
for c in ['y','x','component_key']: cohort[c]=pd.to_numeric(cohort[c],errors='raise').astype(np.int32)
for c in ['p_remove','p_remove_59r2','paper_value']: cohort[c]=pd.to_numeric(cohort[c],errors='raise').astype(np.float32)
cohort['safe_remove']=cohort.safe_remove.astype(bool)
require(not cohort.duplicated(['page','y','x']).any(),'Duplicate page/y/x rows.')
require(cohort.page.nunique()==EXPECTED_PAGES,f'Expected {EXPECTED_PAGES} pages; found {cohort.page.nunique()}.')
require(np.isfinite(cohort[['p_remove','p_remove_59r2','paper_value']].to_numpy()).all(),'Non-finite scored value.')
print('Rows:',f'{len(cohort):,}','Pages:',cohort.page.nunique())

Rows: 12,653,698 Pages: 151


## 4. Resolve observed and clean pages with exact shape/identity checks

In [ ]:
# ============================================================
# Cell 4 - Resolve the 151 annotated-only observed pages
#
# IMPORTANT:
#   This cohort is annotated-only.
#
#   Clean counterparts are NOT required and generally do not
#   exist for these pages.
#
#   This cell therefore:
#       - resolves the authoritative observed page;
#       - verifies coordinate and dimensional consistency;
#       - records exact file/array identities;
#       - records whether a clean counterpart happens to exist;
#       - does NOT fail when clean truth is unavailable.
#
# Clean truth is not fabricated.
# ============================================================


# ------------------------------------------------------------
# 1. Image maps
# ------------------------------------------------------------

clean_maps = [
    image_map(
        ROOT
        / 'source'
        / 'clean'
    ),

    image_map(
        ROOT
        / 'clean'
    ),
]


observed_maps = [
    image_map(
        ROOT
        / 'aligned'
    ),

    image_map(
        ROOT
        / 'source'
        / 'annotated_only'
    ),
]


r60 = (
    Path(
        IDENTITY61[
            'r60'
        ]
    )
    if (
        IDENTITY61.get(
            'r60'
        )
        and Path(
            IDENTITY61[
                'r60'
            ]
        ).exists()
    )
    else None
)


# ------------------------------------------------------------
# 2. Resolution helpers
# ------------------------------------------------------------

def resolve_clean_optional(
    page,
):
    """
    Resolve a clean counterpart only when one genuinely exists.

    Absence is expected for annotated-only pages and is NOT
    an error.
    """

    for mapping in clean_maps:

        if page in mapping:

            return mapping[
                page
            ]

    return None


def resolve_observed(
    page,
):
    """
    Resolve the exact observed page used by the reviewed cohort.

    Prefer Notebook 61 / 60R2 run-local page artefacts before
    falling back to general project folders.
    """

    candidates = [
        NB61
        / 'pages'
        / page
        / 'observed.png',
    ]

    if r60 is not None:

        candidates.extend([
            r60
            / 'pages'
            / page
            / 'observed.png',

            r60
            / 'whole_pages'
            / page
            / 'observed.png',
        ])

    for path in candidates:

        if path.is_file():

            return path


    for mapping in observed_maps:

        if page in mapping:

            return mapping[
                page
            ]

    return None


# ------------------------------------------------------------
# 3. Resolve the complete 151-page observed roster
# ------------------------------------------------------------

page_rows = []

page_runtime = {}

clean_available_pages = []

clean_unavailable_pages = []


for page, group in cohort.groupby(
    'page',
    sort=True,
):

    page = str(
        page
    )

    observed_path = (
        resolve_observed(
            page
        )
    )

    require(
        observed_path is not None,
        (
            f'{page}: authoritative observed page '
            'could not be resolved.'
        ),
    )


    observed = cv2.imread(
        str(
            observed_path
        ),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        observed is not None,
        (
            f'{page}: authoritative observed page '
            f'is unreadable: {observed_path}'
        ),
    )


    # --------------------------------------------------------
    # Validate all cohort coordinates against observed page
    # --------------------------------------------------------

    y = group[
        'y'
    ].to_numpy(
        dtype=np.int32,
        copy=False,
    )

    x = group[
        'x'
    ].to_numpy(
        dtype=np.int32,
        copy=False,
    )


    require(
        (
            y >= 0
        ).all()
        and (
            y < observed.shape[
                0
            ]
        ).all(),
        (
            f'{page}: cohort contains y coordinates '
            'outside the authoritative observed page.'
        ),
    )


    require(
        (
            x >= 0
        ).all()
        and (
            x < observed.shape[
                1
            ]
        ).all(),
        (
            f'{page}: cohort contains x coordinates '
            'outside the authoritative observed page.'
        ),
    )


    # --------------------------------------------------------
    # Clean counterpart is OPTIONAL.
    #
    # If one genuinely exists under the same page identity,
    # record it for later descriptive/audit use.
    #
    # Do NOT assume its absence is an error.
    # --------------------------------------------------------

    clean_path = (
        resolve_clean_optional(
            page
        )
    )

    clean = None

    clean_shape_matches = None

    clean_sha256 = None

    clean_array_sha256 = None


    if clean_path is not None:

        clean = cv2.imread(
            str(
                clean_path
            ),
            cv2.IMREAD_GRAYSCALE,
        )

        require(
            clean is not None,
            (
                f'{page}: a nominal clean counterpart '
                f'exists but is unreadable: {clean_path}'
            ),
        )


        clean_shape_matches = bool(
            clean.shape
            == observed.shape
        )


        # A same-stem clean page is not automatically valid
        # truth for this observed page. Record shape agreement
        # but do not register/resize it here.
        clean_sha256 = (
            sha256_file(
                clean_path
            )
        )

        clean_array_sha256 = (
            array_hash(
                clean
            )
        )


        clean_available_pages.append(
            page
        )

    else:

        clean_unavailable_pages.append(
            page
        )


    # --------------------------------------------------------
    # Runtime state
    #
    # Only observed evidence is authoritative for the
    # annotated-only threshold experiment.
    # --------------------------------------------------------

    page_runtime[
        page
    ] = {
        'observed':
            observed,

        'observed_path':
            observed_path,

        # Optional diagnostic only.
        'clean':
            clean,

        'clean_path':
            clean_path,
    }


    page_rows.append({
        'page':
            page,

        'height':
            int(
                observed.shape[
                    0
                ]
            ),

        'width':
            int(
                observed.shape[
                    1
                ]
            ),

        'cohort_rows':
            int(
                len(
                    group
                )
            ),

        'observed_path':
            str(
                observed_path
            ),

        'observed_sha256':
            sha256_file(
                observed_path
            ),

        'observed_array_sha256':
            array_hash(
                observed
            ),

        'clean_available':
            bool(
                clean_path
                is not None
            ),

        'clean_path':
            (
                str(
                    clean_path
                )
                if clean_path is not None
                else None
            ),

        'clean_shape_matches':
            clean_shape_matches,

        'clean_sha256':
            clean_sha256,

        'clean_array_sha256':
            clean_array_sha256,
    })


    # Do not retain an unnecessary second page image.
    # A genuine optional clean page may be reloaded later if
    # explicitly required for a separate compatible analysis.
    if clean is not None:

        del clean


# ------------------------------------------------------------
# 4. Complete roster checks
# ------------------------------------------------------------

page_resolution = pd.DataFrame(
    page_rows
).sort_values(
    'page'
).reset_index(
    drop=True
)


require(
    len(
        page_resolution
    )
    == EXPECTED_PAGES,
    (
        'Observed-page roster mismatch: '
        f'expected {EXPECTED_PAGES}, '
        f'found {len(page_resolution)}.'
    ),
)


require(
    page_resolution[
        'page'
    ].nunique()
    == EXPECTED_PAGES,
    (
        'Observed-page resolution contains duplicate '
        'page identities.'
    ),
)


require(
    set(
        page_resolution[
            'page'
        ]
    )
    == set(
        cohort[
            'page'
        ].astype(str)
    ),
    (
        'Observed-page resolution does not exactly match '
        'the scored cohort page roster.'
    ),
)


# ------------------------------------------------------------
# 5. Write page-resolution audit
# ------------------------------------------------------------

page_resolution.to_csv(
    AUDIT
    / 'page_pair_resolution.csv',
    index=False,
)


clean_truth_status = {
    'cohort_role':
        'annotated_only reviewed operational cohort',

    'pages':
        int(
            EXPECTED_PAGES
        ),

    'observed_pages_resolved':
        int(
            len(
                page_resolution
            )
        ),

    'clean_pages_required':
        False,

    'clean_pages_available_same_stem':
        int(
            len(
                clean_available_pages
            )
        ),

    'clean_pages_unavailable':
        int(
            len(
                clean_unavailable_pages
            )
        ),

    'clean_available_pages':
        sorted(
            clean_available_pages
        ),

    'clean_unavailable_pages':
        sorted(
            clean_unavailable_pages
        ),

    'clean_truth_complete_for_151_page_cohort':
        bool(
            len(
                clean_available_pages
            )
            == EXPECTED_PAGES
        ),

    'truth_policy':
        (
            'Do not fabricate clean truth. '
            'The 151-page annotated-only cohort supports '
            'observable and visual threshold analysis. '
            'Truth-side safety requires a separate cohort '
            'with genuine registered clean counterparts.'
        ),
}


atomic_json(
    AUDIT
    / 'clean_truth_availability.json',
    clean_truth_status,
)


# ------------------------------------------------------------
# 6. Console summary
# ------------------------------------------------------------

print(
    'Resolved annotated-only observed pages:',
    len(
        page_resolution
    ),
)

print(
    'Pages with same-stem clean counterpart:',
    len(
        clean_available_pages
    ),
)

print(
    'Pages without clean counterpart:',
    len(
        clean_unavailable_pages
    ),
)


if clean_available_pages:

    print(
        '\nSame-stem clean counterparts exist for:',
        clean_available_pages[
            :20
        ],
    )

    if len(
        clean_available_pages
    ) > 20:

        print(
            '... plus',
            len(
                clean_available_pages
            )
            - 20,
            'more.'
        )


if clean_unavailable_pages:

    print(
        '\nClean truth is unavailable for the full '
        '151-page cohort, as expected for annotated-only '
        'runtime pages.'
    )


print(
    '\nCELL 4 STATUS: PASS'
)

print(
    'Observed-page 

## 5. Rebuild and lock all diagnostic decisions before loading truth labels

In [ ]:
def coherent_decision(y,x,probability,height,width,threshold):
    y=np.asarray(y,np.int32); x=np.asarray(x,np.int32); probability=np.asarray(probability,np.float32)
    provisional=probability>=float(threshold)
    candidate=np.zeros((height,width),np.uint8); candidate[y[provisional],x[provisional]]=1
    n,labels,stats,_=cv2.connectedComponentsWithStats(candidate,CONNECTIVITY)
    pmap=np.full((height,width),np.nan,np.float32); pmap[y,x]=probability
    safe=np.zeros((height,width),bool)
    for cid in range(1,n):
        region=labels==cid; size=int(stats[cid,cv2.CC_STAT_AREA]); vals=pmap[region]; finite=vals[np.isfinite(vals)]
        if size>=MINIMUM_RUN and len(finite)==size and float(finite.min())>=threshold and float(finite.mean())>=threshold: safe[region]=True
    return safe[y,x]

for t in THRESHOLDS:
    final=np.zeros(len(cohort),bool)
    for page,idx in cohort.groupby('page',sort=False).groups.items():
        ii=np.asarray(list(idx),dtype=np.int64); z=page_runtime[page]
        final[ii]=coherent_decision(cohort.loc[ii,'y'],cohort.loc[ii,'x'],cohort.loc[ii,'p_remove_59r2'],z['observed'].shape[0],z['observed'].shape[1],t)
    cohort[f'safe_{t:.3f}']=final

# Frozen replay checks.
if (NB61/'shadow_test_summary.json').is_file():
    s61=json.loads((NB61/'shadow_test_summary.json').read_text())
    require(int(cohort['safe_0.995'].sum())==int(s61['59r2_safe_remove']),'0.995 aggregate does not reproduce Notebook 61.')
focus=cohort[cohort.page==FOCUS_PAGE]
if (NB61/'pages'/FOCUS_PAGE/'59r2_shadow.png').is_file():
    obs=page_runtime[FOCUS_PAGE]['observed']; replay=obs.copy(); s=focus.safe_0.995.to_numpy(); replay[focus.y.to_numpy()[s],focus.x.to_numpy()[s]]=np.clip(np.rint(focus.paper_value.to_numpy()[s]),0,255).astype(np.uint8)
    saved=cv2.imread(str(NB61/'pages'/FOCUS_PAGE/'59r2_shadow.png'),0); require(np.array_equal(replay,saved),'Page258 frozen shadow replay failed.')

decision_cols=['page','y','x']+[f'safe_{t:.3f}' for t in THRESHOLDS]
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'rows':len(cohort),'pages':cohort.page.nunique(),'thresholds':THRESHOLDS,'decision_semantic_sha256':semantic_table_hash(cohort,decision_cols),'source_parquet_sha256':sha256_file(PARQUET_PATH),'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'clean_truth_loaded_into_decisions':False}
atomic_json(AUDIT/'decision_lock_before_truth.json',lock)
print(json.dumps(lock,indent=2))

## 6. Construct exact page/y/x truth labels from clean pages

In [ ]:
# ============================================================
# Cell 6 - Truth availability record for the annotated-only
#          151-page cohort
#
# IMPORTANT:
#   The reviewed Notebook 61 cohort is annotated-only.
#
#   Clean counterparts are NOT required at runtime and do not
#   generally exist for these pages.
#
#   Therefore this notebook MUST NOT fabricate:
#
#       PRINT
#       ANNOTATION_NONPRINT
#       UNCERTAIN
#
#   clean-derived truth labels for the complete 151-page
#   population.
#
#   This cell:
#       1. verifies the decision population is already locked;
#       2. records exactly which pages happen to have optional
#          clean counterparts;
#       3. declares clean-truth safety unavailable for the
#          complete 151-page cohort;
#       4. creates compatibility variables/tables so later
#          diagnostic cells can continue;
#       5. does not load clean pixels into the scored cohort.
#
# Threshold ladders, probability analysis, component analysis
# and the Page258 top-right diagnostic remain valid observable
# analyses.
# ============================================================


# ------------------------------------------------------------
# 1. Require the pre-truth decision lock
# ------------------------------------------------------------

DECISION_LOCK_PATH = (
    AUDIT
    / 'decision_lock_before_truth.json'
)

require(
    DECISION_LOCK_PATH.is_file(),
    (
        'Decision lock is missing. '
        'Diagnostic decisions must be rebuilt and locked '
        'before truth availability is assessed.'
    ),
)

decision_lock = json.loads(
    DECISION_LOCK_PATH.read_text()
)

require(
    int(
        decision_lock[
            'rows'
        ]
    )
    == len(
        cohort
    ),
    (
        'Decision-lock row count does not match '
        'the current cohort.'
    ),
)

require(
    int(
        decision_lock[
            'pages'
        ]
    )
    == EXPECTED_PAGES,
    (
        'Decision-lock page count does not match '
        f'the expected {EXPECTED_PAGES}-page cohort.'
    ),
)


# ------------------------------------------------------------
# 2. Verify that the locked decisions have not changed
# ------------------------------------------------------------

decision_cols = (
    [
        'page',
        'y',
        'x',
    ]
    + [
        f'safe_{threshold:.3f}'
        for threshold in THRESHOLDS
    ]
)

current_decision_hash = (
    semantic_table_hash(
        cohort,
        decision_cols,
    )
)

require(
    current_decision_hash
    == decision_lock[
        'decision_semantic_sha256'
    ],
    (
        'Diagnostic decisions changed after the '
        'pre-truth decision lock.'
    ),
)


# ------------------------------------------------------------
# 3. Summarise optional clean-counterpart availability
#
# Cell 4 already resolved these on a best-effort basis.
#
# A same-stem clean file is NOT automatically treated as valid
# truth. It is recorded only as optional evidence for a future
# separately commissioned registered-clean analysis.
# ------------------------------------------------------------

require(
    'page_resolution'
    in globals(),
    (
        'Cell 4 page_resolution table is unavailable.'
    ),
)

require(
    len(
        page_resolution
    )
    == EXPECTED_PAGES,
    (
        'Cell 4 page-resolution roster is incomplete.'
    ),
)

require(
    'clean_available'
    in page_resolution.columns,
    (
        'Cell 4 page_resolution lacks clean_available.'
    ),
)


clean_available = (
    page_resolution[
        page_resolution[
            'clean_available'
        ]
    ]
    .copy()
)

clean_unavailable = (
    page_resolution[
        ~page_resolution[
            'clean_available'
        ]
    ]
    .copy()
)


clean_shape_compatible = (
    clean_available[
        clean_available[
            'clean_shape_matches'
        ]
        == True
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. Explicit truth status
#
# Full-cohort clean truth is intentionally unavailable unless
# every page has a separately validated registered clean
# counterpart.
#
# We do NOT silently use a partial subset for the 151-page
# threshold comparison because that would change the cohort
# being evaluated.
# ------------------------------------------------------------

FULL_COHORT_CLEAN_TRUTH_AVAILABLE = bool(
    len(
        clean_shape_compatible
    )
    == EXPECTED_PAGES
)


# Even if every page happened to have a same-stem clean file,
# Notebook 64 still has not established registration identity
# for those files. Therefore same-stem availability alone is
# insufficient to unlock clean truth.
REGISTERED_CLEAN_TRUTH_AVAILABLE = False


TRUTH_SCORING_AVAILABLE = bool(
    FULL_COHORT_CLEAN_TRUTH_AVAILABLE
    and REGISTERED_CLEAN_TRUTH_AVAILABLE
)


truth_status = {
    'cohort':
        'Notebook 61 reviewed annotated-only cohort',

    'cohort_pages':
        int(
            EXPECTED_PAGES
        ),

    'cohort_rows':
        int(
            len(
                cohort
            )
        ),

    'clean_truth_required_for_runtime':
        False,

    'same_stem_clean_pages_found':
        int(
            len(
                clean_available
            )
        ),

    'same_stem_clean_pages_shape_compatible':
        int(
            len(
                clean_shape_compatible
            )
        ),

    'clean_pages_absent':
        int(
            len(
                clean_unavailable
            )
        ),

    'full_cohort_same_stem_clean_available':
        bool(
            FULL_COHORT_CLEAN_TRUTH_AVAILABLE
        ),

    'registered_clean_truth_available':
        bool(
            REGISTERED_CLEAN_TRUTH_AVAILABLE
        ),

    'truth_scoring_available':
        bool(
            TRUTH_SCORING_AVAILABLE
        ),

    'truth_table_created':
        False,

    'truth_labels_fabricated':
        False,

    'clean_pixels_loaded_into_cohort':
        False,

    'permitted_151_page_analysis': [
        'threshold coverage',
        'page concentration',
        'component concentration',
        'probability distributions',
        'observed intensity',
        'paper-value difference',
        'page-global coherence',
        '59R1 comparison',
        'whole-page threshold ladders',
        'Page258 top-right diagnostic',
        'candidate-population coverage',
    ],

    'not_permitted_without_registered_clean_truth': [
        'clean PRINT labels',
        'clean ANNOTATION_NONPRINT labels',
        'clean truth UNCERTAIN labels',
        'false removal of genuine printed ink',
        'truth-side false-removal rate',
        'truth-side safety promotion claim',
    ],

    'scientific_interpretation': (
        'The complete 151-page cohort supports observable, '
        'probability, concentration and visual threshold '
        'analysis. It does not support clean-truth safety '
        'claims because registered clean counterparts are '
        'not available for the complete cohort.'
    ),

    'required_follow_up_for_truth_safety': (
        'Use a separately declared matched clean/annotated '
        'validation cohort with registered clean truth. '
        'Do not manufacture truth for annotated-only pages.'
    ),
}


atomic_json(
    AUDIT
    / 'truth_availability.json',
    truth_status,
)


# ------------------------------------------------------------
# 5. Persist page-level availability roster
# ------------------------------------------------------------

truth_availability_by_page = (
    page_resolution[
        [
            'page',
            'height',
            'width',
            'cohort_rows',
            'observed_path',
            'observed_sha256',
            'clean_available',
            'clean_path',
            'clean_shape_matches',
            'clean_sha256',
        ]
    ]
    .copy()
)

truth_availability_by_page[
    'registered_clean_truth'
] = False

truth_availability_by_page[
    'truth_status'
] = np.where(
    truth_availability_by_page[
        'clean_available'
    ],
    (
        'same_stem_clean_found_but_not_registered_'
        'for_this_annotated_only_truth_experiment'
    ),
    'clean_counterpart_unavailable',
)


truth_availability_by_page.to_csv(
    TABLES
    / 'truth_availability_by_page.csv',
    index=False,
)


# ------------------------------------------------------------
# 6. Create compatibility variables for later cells
#
# Later Notebook 64 cells should test:
#
#       if TRUTH_SCORING_AVAILABLE:
#
# before attempting truth-side safety calculations.
#
# Do not use a synthetic truth_class column.
# ------------------------------------------------------------

truth_table = None

truth_identity = {
    'available':
        False,

    'rows':
        0,

    'pages':
        0,

    'semantic_sha256':
        None,

    'reason':
        (
            'Complete registered clean truth is unavailable '
            'for the 151-page annotated-only cohort.'
        ),

    'fabricated':
        False,
}


atomic_json(
    AUDIT
    / 'truth_table_identity.json',
    truth_identity,
)


# ------------------------------------------------------------
# 7. Human-readable truth status file
# ------------------------------------------------------------

truth_lines = [
    'NOTEBOOK 64 CLEAN-TRUTH STATUS',
    '',

    'Cohort role:',
    'Reviewed annotated-only operational cohort',
    '',

    f'Cohort pages: {EXPECTED_PAGES}',
    f'Cohort rows: {len(cohort):,}',

    (
        'Same-stem clean pages found: '
        f'{len(clean_available)}'
    ),

    (
        'Same-stem clean pages with matching dimensions: '
        f'{len(clean_shape_compatible)}'
    ),

    (
        'Pages without a clean counterpart: '
        f'{len(clean_unavailable)}'
    ),

    '',

    'REGISTERED CLEAN TRUTH AVAILABLE: FALSE',
    'TRUTH-SIDE SAFETY AVAILABLE: FALSE',

    '',

    (
        'The annotated-only threshold experiment may continue '
        'using observable evidence and whole-page visual '
        'diagnostics.'
    ),

    (
        'No PRINT / ANNOTATION_NONPRINT / UNCERTAIN clean '
        'truth labels have been fabricated.'
    ),

    (
        'Truth-side false-removal safety must be evaluated '
        'separately on a genuinely matched and registered '
        'clean-reference cohort.'
    ),
]


(
    AUDIT
    / 'truth_availability.txt'
).write_text(
    '\n'.join(
        truth_lines
    )
    + '\n'
)


# ------------------------------------------------------------
# 8. Console summary
# ------------------------------------------------------------

print(
    '\n'.join(
        truth_lines
    )
)

print(
    '\nCELL 6 STATUS: PASS'
)

print(
    'Observable/visual analysis may continue.'
)

print(
    'Clean-truth scoring is disabled for this cohort.'
)

## 7. Compare 0.995, 0.993 and 0.985 on removal safety

In [ ]:
# ============================================================
# Cell 7 - Truth-side safety status
# ============================================================

truth_safety_summary = pd.DataFrame([
    {
        'threshold':
            threshold,

        'selected_pixels':
            int(
                cohort[
                    f'safe_{threshold:.3f}'
                ].sum()
            ),

        'clean_truth_available':
            False,

        'false_remove_print':
            np.nan,

        'correct_remove_annotation':
            np.nan,

        'truth_uncertain_selected':
            np.nan,

        'false_removal_rate':
            np.nan,

        'wilson95_upper_false_removal':
            np.nan,

        'interpretation':
            (
                'Clean-truth safety is unavailable for '
                'the annotated-only cohort.'
            ),
    }

    for threshold in SAFETY_THRESHOLDS
])

truth_safety_summary.to_csv(
    TABLES
    / 'truth_safety_threshold_summary.csv',
    index=False,
)

atomic_json(
    AUDIT
    / 'truth_safety_status.json',
    {
        'available':
            False,

        'reason':
            (
                'The complete 151-page annotated-only cohort '
                'does not have registered clean counterparts.'
            ),

        'thresholds':
            SAFETY_THRESHOLDS,

        'promotion_authority':
            False,
    },
)

display(
    truth_safety_summary
)

## 8. Truth-stratified probability distribution and threshold transitions

In [ ]:
# ============================================================
# Cell 8 - Observable threshold-transition diagnostics
# ============================================================

transition_rows = []

for higher, lower in zip(
    THRESHOLDS[:-1],
    THRESHOLDS[1:],
):

    higher_col = (
        f'safe_{higher:.3f}'
    )

    lower_col = (
        f'safe_{lower:.3f}'
    )

    newly_selected = (
        cohort[
            lower_col
        ]
        & ~cohort[
            higher_col
        ]
    )

    selected = (
        cohort[
            newly_selected
        ]
        .copy()
    )

    transition_rows.append({
        'from_threshold':
            higher,

        'to_threshold':
            lower,

        'new_pixels':
            int(
                newly_selected.sum()
            ),

        'pages_gaining':
            int(
                selected[
                    'page'
                ].nunique()
            ),

        'components_gaining':
            int(
                selected[
                    [
                        'page',
                        'component_key',
                    ]
                ]
                .drop_duplicates()
                .shape[
                    0
                ]
            ),

        'mean_p_remove_59r2':
            (
                float(
                    selected[
                        'p_remove_59r2'
                    ].mean()
                )
                if len(
                    selected
                )
                else np.nan
            ),

        'median_p_remove_59r2':
            (
                float(
                    selected[
                        'p_remove_59r2'
                    ].median()
                )
                if len(
                    selected
                )
                else np.nan
            ),

        'observed_ge_180':
            int(
                (
                    selected[
                        'observed_value'
                    ]
                    >= 180
                ).sum()
            ),

        'observed_ge_210':
            int(
                (
                    selected[
                        'observed_value'
                    ]
                    >= 210
                ).sum()
            ),

        'overlap_with_59r1_safe_remove':
            int(
                selected[
                    'safe_remove'
                ].sum()
            ),

        'not_selected_by_59r1':
            int(
                (
                    ~selected[
                        'safe_remove'
                    ]
                ).sum()
            ),
    })


threshold_transitions = pd.DataFrame(
    transition_rows
)

threshold_transitions.to_csv(
    TABLES
    / 'observable_threshold_transitions.csv',
    index=False,
)

display(
    threshold_transitions
)

## 9. Generate a seven-panel threshold ladder for every page

In [ ]:
def render_page(page,t):
    g=cohort[cohort.page==page]; obs=page_runtime[page]['observed']; out=obs.copy(); s=g[f'safe_{t:.3f}'].to_numpy(); yy=g.y.to_numpy(); xx=g.x.to_numpy(); paper=g.paper_value.to_numpy(); out[yy[s],xx[s]]=np.clip(np.rint(paper[s]),0,255).astype(np.uint8); return out,int(s.sum())

def make_ladder(page):
    fig,axes=plt.subplots(2,4,figsize=(20,14)); axes=axes.flat
    counts=[]
    for i,t in enumerate(THRESHOLDS):
        out,n=render_page(page,t); counts.append({'page':page,'threshold':t,'removed':n}); axes[i].imshow(out,cmap='gray',vmin=0,vmax=255); axes[i].set_title(f'{t:.3f}: {n:,} removed',fontsize=14); axes[i].axis('off')
    axes[7].axis('off'); plt.tight_layout(); path=LADDERS/f'{page}_threshold_ladder.png'; plt.savefig(path,dpi=145,bbox_inches='tight'); plt.close(fig); return path,counts

ladder_rows=[]
for i,page in enumerate(sorted(cohort.page.unique()),1):
    path,rows=make_ladder(page); ladder_rows.extend(rows); print(f'{i:03d}/{EXPECTED_PAGES}: {path.name}')
pd.DataFrame(ladder_rows).to_csv(TABLES/'all_page_ladder_counts.csv',index=False)

# Contact sheets are deliberately split to remain readable.
pages=sorted(cohort.page.unique()); batch_size=12
for start in range(0,len(pages),batch_size):
    batch=pages[start:start+batch_size]; fig,axes=plt.subplots(len(batch),len(THRESHOLDS),figsize=(3.6*len(THRESHOLDS),4.8*len(batch)),squeeze=False)
    for r,page in enumerate(batch):
        for c,t in enumerate(THRESHOLDS):
            out,n=render_page(page,t); axes[r,c].imshow(out,cmap='gray',vmin=0,vmax=255); axes[r,c].set_title(f'{page} {t:.3f}
{n:,}',fontsize=8); axes[r,c].axis('off')
    plt.tight_layout(); plt.savefig(CONTACTS/f'all_pages_threshold_ladders_{start+1:03d}_{start+len(batch):03d}.png',dpi=100,bbox_inches='tight'); plt.close(fig)
print('Individual ladders:',len(list(LADDERS.glob('*.png'))))

## 10. Diagnose Page258 “SCORE + 27 parts” top-right region

In [ ]:
# ============================================================
# Cell 10 - Diagnose the Page258 top-right region containing
#           "SCORE + 27 parts"
#
# This is an observable-only diagnostic.
#
# The cell answers:
#
#   1. Is the handwriting inside the scored candidate
#      population?
#
#   2. If present in the candidate population, what
#      p_remove_59r2 values does the region receive?
#
#   3. Which diagnostic thresholds select those rows?
#
#   4. Is threshold failure or page-global coherence
#      responsible for retention?
#
#   5. Which scored components intersect the region?
#
# No clean page or clean-derived truth is used.
# ============================================================


# ------------------------------------------------------------
# 1. Validate prerequisites
# ------------------------------------------------------------

require(
    FOCUS_PAGE
    in set(
        cohort[
            'page'
        ].astype(str)
    ),
    (
        f'{FOCUS_PAGE} is not present in the scored cohort.'
    ),
)


require(
    FOCUS_PAGE
    in page_runtime,
    (
        f'{FOCUS_PAGE} is not present in page_runtime.'
    ),
)


require(
    'observed'
    in page_runtime[
        FOCUS_PAGE
    ],
    (
        f'{FOCUS_PAGE}: authoritative observed image '
        'is unavailable.'
    ),
)


required_focus_columns = {
    'page',
    'y',
    'x',
    'component_key',
    'p_remove_59r2',
    'paper_value',
    'safe_remove',
}


for threshold in THRESHOLDS:

    required_focus_columns.add(
        f'safe_{threshold:.3f}'
    )


missing_focus_columns = (
    required_focus_columns
    - set(
        cohort.columns
    )
)


require(
    not missing_focus_columns,
    (
        'Page258 diagnostic is missing cohort columns: '
        f'{sorted(missing_focus_columns)}'
    ),
)


# ------------------------------------------------------------
# 2. Load Page258 observed image and rows
# ------------------------------------------------------------

observed = page_runtime[
    FOCUS_PAGE
][
    'observed'
]


require(
    observed is not None,
    (
        f'{FOCUS_PAGE}: observed image is None.'
    ),
)


require(
    observed.ndim
    == 2,
    (
        f'{FOCUS_PAGE}: expected a greyscale observed image.'
    ),
)


height, width = (
    observed.shape
)


focus_rows = (
    cohort[
        cohort[
            'page'
        ].astype(str)
        == FOCUS_PAGE
    ]
    .copy()
    .reset_index(
        drop=False
    )
    .rename(
        columns={
            'index':
                'cohort_index'
        }
    )
)


require(
    len(
        focus_rows
    )
    > 0,
    (
        f'{FOCUS_PAGE}: no scored rows were found.'
    ),
)


focus_y = (
    focus_rows[
        'y'
    ]
    .to_numpy(
        dtype=np.int32,
        copy=False,
    )
)


focus_x = (
    focus_rows[
        'x'
    ]
    .to_numpy(
        dtype=np.int32,
        copy=False,
    )
)


require(
    (
        focus_y
        >= 0
    ).all()
    and (
        focus_y
        < height
    ).all()
    and (
        focus_x
        >= 0
    ).all()
    and (
        focus_x
        < width
    ).all(),
    (
        f'{FOCUS_PAGE}: one or more scored coordinates '
        'fall outside the observed image.'
    ),
)


focus_rows[
    'observed_value'
] = observed[
    focus_y,
    focus_x,
].astype(
    np.uint8
)


# ------------------------------------------------------------
# 3. Resolve the top-right diagnostic crop
#
# If PAGE258_TOP_RIGHT_CROP is None, use a broad upper-right
# crop. For a tighter result, set exact coordinates in Cell 1:
#
# PAGE258_TOP_RIGHT_CROP = (x0, y0, x1, y1)
# ------------------------------------------------------------

if PAGE258_TOP_RIGHT_CROP is None:

    crop_x0 = int(
        round(
            width
            * 0.66
        )
    )

    crop_y0 = 0

    crop_x1 = width

    crop_y1 = int(
        round(
            height
            * 0.20
        )
    )

else:

    require(
        len(
            PAGE258_TOP_RIGHT_CROP
        )
        == 4,
        (
            'PAGE258_TOP_RIGHT_CROP must contain '
            '(x0, y0, x1, y1).'
        ),
    )

    (
        crop_x0,
        crop_y0,
        crop_x1,
        crop_y1,
    ) = [
        int(
            value
        )
        for value
        in PAGE258_TOP_RIGHT_CROP
    ]


crop_x0 = max(
    0,
    crop_x0,
)

crop_y0 = max(
    0,
    crop_y0,
)

crop_x1 = min(
    width,
    crop_x1,
)

crop_y1 = min(
    height,
    crop_y1,
)


require(
    crop_x1
    > crop_x0
    and crop_y1
    > crop_y0,
    (
        'Invalid Page258 top-right crop: '
        f'{crop_x0}, {crop_y0}, {crop_x1}, {crop_y1}'
    ),
)


crop_width = (
    crop_x1
    - crop_x0
)

crop_height = (
    crop_y1
    - crop_y0
)

crop_image_pixels = int(
    crop_width
    * crop_height
)


inside_crop = (
    (
        focus_rows[
            'x'
        ]
        >= crop_x0
    )
    & (
        focus_rows[
            'x'
        ]
        < crop_x1
    )
    & (
        focus_rows[
            'y'
        ]
        >= crop_y0
    )
    & (
        focus_rows[
            'y'
        ]
        < crop_y1
    )
)


crop_rows = (
    focus_rows[
        inside_crop
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 4. Assign probability bands
# ------------------------------------------------------------

probability_band_definitions = [
    {
        'band_id':
            'A_GE_0995',

        'label':
            '>= 0.995',

        'low':
            0.995,

        'high':
            1.0000001,

        'bgr':
            (
                0,
                190,
                0,
            ),
    },

    {
        'band_id':
            'B_0993_0995',

        'label':
            '0.993-0.995',

        'low':
            0.993,

        'high':
            0.995,

        'bgr':
            (
                0,
                255,
                100,
            ),
    },

    {
        'band_id':
            'C_0990_0993',

        'label':
            '0.990-0.993',

        'low':
            0.990,

        'high':
            0.993,

        'bgr':
            (
                0,
                255,
                255,
            ),
    },

    {
        'band_id':
            'D_0985_0990',

        'label':
            '0.985-0.990',

        'low':
            0.985,

        'high':
            0.990,

        'bgr':
            (
                0,
                165,
                255,
            ),
    },

    {
        'band_id':
            'E_0980_0985',

        'label':
            '0.980-0.985',

        'low':
            0.980,

        'high':
            0.985,

        'bgr':
            (
                255,
                180,
                0,
            ),
    },

    {
        'band_id':
            'F_0975_0980',

        'label':
            '0.975-0.980',

        'low':
            0.975,

        'high':
            0.980,

        'bgr':
            (
                255,
                80,
                160,
            ),
    },

    {
        'band_id':
            'G_0950_0975',

        'label':
            '0.950-0.975',

        'low':
            0.950,

        'high':
            0.975,

        'bgr':
            (
                180,
                0,
                255,
            ),
    },

    {
        'band_id':
            'H_LT_0950',

        'label':
            '< 0.950',

        'low':
            0.000,

        'high':
            0.950,

        'bgr':
            (
                80,
                80,
                80,
            ),
    },
]


def assign_probability_band(
    probability,
):

    probability = float(
        probability
    )

    for definition in probability_band_definitions:

        if (
            definition[
                'low'
            ]
            <= probability
            < definition[
                'high'
            ]
        ):

            return definition[
                'band_id'
            ]

    return 'UNASSIGNED'


crop_rows[
    'probability_band'
] = [
    assign_probability_band(
        value
    )
    for value
    in crop_rows[
        'p_remove_59r2'
    ]
]


require(
    not (
        crop_rows[
            'probability_band'
        ]
        == 'UNASSIGNED'
    ).any(),
    (
        'One or more crop rows could not be assigned '
        'to a probability band.'
    ),
)


# ------------------------------------------------------------
# 5. Measure threshold and coherence behaviour
# ------------------------------------------------------------

for threshold in THRESHOLDS:

    threshold_label = (
        f'{threshold:.3f}'
    )

    safe_column = (
        f'safe_{threshold_label}'
    )

    probability_column = (
        f'passes_probability_{threshold_label}'
    )

    coherence_column = (
        f'coherence_rejected_{threshold_label}'
    )


    crop_rows[
        probability_column
    ] = (
        crop_rows[
            'p_remove_59r2'
        ]
        >= threshold
    )


    crop_rows[
        coherence_column
    ] = (
        crop_rows[
            probability_column
        ]
        & ~crop_rows[
            safe_column
        ]
    )


# ------------------------------------------------------------
# 6. Candidate-population mask
#
# White pixels mean that an exact Page258 coordinate exists
# in the compact scored cohort.
#
# Black pixels were not represented by a scored row and cannot
# be selected by any threshold in this experiment.
# ------------------------------------------------------------

candidate_population = np.zeros(
    observed.shape,
    dtype=bool,
)


candidate_population[
    focus_y,
    focus_x,
] = True


candidate_crop = (
    candidate_population[
        crop_y0:crop_y1,
        crop_x0:crop_x1,
    ]
)


candidate_population_png = np.where(
    candidate_crop,
    255,
    0,
).astype(
    np.uint8
)


candidate_population_path = (
    FOCUS
    / 'top_right_candidate_population.png'
)


require(
    cv2.imwrite(
        str(
            candidate_population_path
        ),
        candidate_population_png,
    ),
    (
        'Failed to write top-right candidate-population image.'
    ),
)


# ------------------------------------------------------------
# 7. Probability-band overlay
# ------------------------------------------------------------

observed_crop = (
    observed[
        crop_y0:crop_y1,
        crop_x0:crop_x1,
    ]
)


base_bgr = cv2.cvtColor(
    observed_crop,
    cv2.COLOR_GRAY2BGR,
)


colour_layer = (
    base_bgr.copy()
)


active_probability_pixels = np.zeros(
    observed_crop.shape,
    dtype=bool,
)


band_summary_rows = []


for definition in probability_band_definitions:

    band_id = (
        definition[
            'band_id'
        ]
    )

    selected_band_rows = (
        crop_rows[
            crop_rows[
                'probability_band'
            ]
            == band_id
        ]
    )


    local_mask = np.zeros(
        observed_crop.shape,
        dtype=bool,
    )


    if len(
        selected_band_rows
    ):

        local_y = (
            selected_band_rows[
                'y'
            ]
            .to_numpy(
                dtype=np.int32
            )
            - crop_y0
        )

        local_x = (
            selected_band_rows[
                'x'
            ]
            .to_numpy(
                dtype=np.int32
            )
            - crop_x0
        )


        local_mask[
            local_y,
            local_x,
        ] = True


        colour_layer[
            local_mask
        ] = definition[
            'bgr'
        ]


        active_probability_pixels |= (
            local_mask
        )


    band_summary_rows.append({
        'band_id':
            band_id,

        'label':
            definition[
                'label'
            ],

        'low':
            definition[
                'low'
            ],

        'high':
            definition[
                'high'
            ],

        'pixels':
            int(
                len(
                    selected_band_rows
                )
            ),

        'components':
            int(
                selected_band_rows[
                    'component_key'
                ].nunique()
            )
            if len(
                selected_band_rows
            )
            else 0,

        'mean_observed_value':
            float(
                selected_band_rows[
                    'observed_value'
                ].mean()
            )
            if len(
                selected_band_rows
            )
            else np.nan,
    })


probability_overlay = (
    base_bgr.copy()
)


probability_overlay[
    active_probability_pixels
] = cv2.addWeighted(
    base_bgr,
    0.38,
    colour_layer,
    0.62,
    0,
)[
    active_probability_pixels
]


# Add a white banner and legend.

banner_height = 105


probability_canvas = np.full(
    (
        probability_overlay.shape[
            0
        ]
        + banner_height,

        probability_overlay.shape[
            1
        ],

        3,
    ),
    255,
    dtype=np.uint8,
)


probability_canvas[
    banner_height:
] = probability_overlay


cv2.putText(
    probability_canvas,
    (
        f'{FOCUS_PAGE} top-right: '
        '59R2 p_remove probability bands'
    ),
    (
        12,
        28,
    ),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.58,
    (
        30,
        30,
        30,
    ),
    1,
    cv2.LINE_AA,
)


legend_x = 12

legend_y = 62


for definition in probability_band_definitions:

    cv2.rectangle(
        probability_canvas,
        (
            legend_x,
            legend_y
            - 15,
        ),
        (
            legend_x
            + 18,
            legend_y
            + 3,
        ),
        definition[
            'bgr'
        ],
        -1,
    )


    cv2.putText(
        probability_canvas,
        definition[
            'label'
        ],
        (
            legend_x
            + 24,
            legend_y
            + 2,
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.34,
        (
            30,
            30,
            30,
        ),
        1,
        cv2.LINE_AA,
    )


    legend_x += 118


    if (
        legend_x
        + 115
        >= probability_canvas.shape[
            1
        ]
    ):

        legend_x = 12

        legend_y += 31


probability_overlay_path = (
    FOCUS
    / 'top_right_probability_overlay.png'
)


require(
    cv2.imwrite(
        str(
            probability_overlay_path
        ),
        probability_canvas,
    ),
    (
        'Failed to write top-right probability overlay.'
    ),
)


# ------------------------------------------------------------
# 8. Threshold crop ladder
# ------------------------------------------------------------

figure, axes = plt.subplots(
    2,
    4,
    figsize=(
        18,
        8,
    ),
)


axes = (
    axes.flat
)


for index, threshold in enumerate(
    THRESHOLDS
):

    safe_column = (
        f'safe_{threshold:.3f}'
    )


    safe = (
        focus_rows[
            safe_column
        ]
        .to_numpy(
            dtype=bool
        )
    )


    rendered = (
        observed.copy()
    )


    paper_values = (
        focus_rows[
            'paper_value'
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    rendered[
        focus_y[
            safe
        ],
        focus_x[
            safe
        ],
    ] = np.clip(
        np.rint(
            paper_values[
                safe
            ]
        ),
        0,
        255,
    ).astype(
        np.uint8
    )


    crop_safe_count = int(
        crop_rows[
            safe_column
        ].sum()
    )


    axes[
        index
    ].imshow(
        rendered[
            crop_y0:crop_y1,
            crop_x0:crop_x1,
        ],
        cmap='gray',
        vmin=0,
        vmax=255,
    )


    axes[
        index
    ].set_title(
        (
            f'{threshold:.3f}: '
            f'{crop_safe_count:,} crop rows removed'
        ),
        fontsize=11,
    )


    axes[
        index
    ].axis(
        'off'
    )


axes[
    7
].axis(
    'off'
)


plt.tight_layout()


top_right_ladder_path = (
    FOCUS
    / 'top_right_score_27_parts_threshold_ladder.png'
)


plt.savefig(
    top_right_ladder_path,
    dpi=180,
    bbox_inches='tight',
)


plt.close(
    figure
)


# ------------------------------------------------------------
# 9. Probability distribution table
# ------------------------------------------------------------

probability_distribution = (
    pd.DataFrame(
        band_summary_rows
    )
)


probability_distribution[
    'fraction_of_candidate_rows'
] = (
    probability_distribution[
        'pixels'
    ]
    / max(
        len(
            crop_rows
        ),
        1,
    )
)


probability_distribution.to_csv(
    FOCUS
    / 'top_right_probability_distribution.csv',
    index=False,
)


# ------------------------------------------------------------
# 10. Component-level diagnostic
# ------------------------------------------------------------

component_diagnostic_columns = [
    'component_key',
    'candidate_rows',
    'y_min',
    'y_max',
    'x_min',
    'x_max',
    'mean_probability',
    'median_probability',
    'maximum_probability',
    'mean_observed_value',
    'minimum_observed_value',
    'maximum_observed_value',
]


if len(
    crop_rows
):

    aggregation = {
        'candidate_rows':
            (
                'p_remove_59r2',
                'size',
            ),

        'y_min':
            (
                'y',
                'min',
            ),

        'y_max':
            (
                'y',
                'max',
            ),

        'x_min':
            (
                'x',
                'min',
            ),

        'x_max':
            (
                'x',
                'max',
            ),

        'mean_probability':
            (
                'p_remove_59r2',
                'mean',
            ),

        'median_probability':
            (
                'p_remove_59r2',
                'median',
            ),

        'maximum_probability':
            (
                'p_remove_59r2',
                'max',
            ),

        'mean_observed_value':
            (
                'observed_value',
                'mean',
            ),

        'minimum_observed_value':
            (
                'observed_value',
                'min',
            ),

        'maximum_observed_value':
            (
                'observed_value',
                'max',
            ),
    }


    component_diagnostic = (
        crop_rows
        .groupby(
            'component_key',
            as_index=False,
        )
        .agg(
            **aggregation
        )
    )


    for threshold in THRESHOLDS:

        safe_column = (
            f'safe_{threshold:.3f}'
        )

        component_counts = (
            crop_rows
            .groupby(
                'component_key'
            )[
                safe_column
            ]
            .sum()
        )


        component_diagnostic[
            f'selected_{threshold:.3f}'
        ] = (
            component_diagnostic[
                'component_key'
            ]
            .map(
                component_counts
            )
            .fillna(
                0
            )
            .astype(
                int
            )
        )


    component_diagnostic = (
        component_diagnostic
        .sort_values(
            [
                'candidate_rows',
                'maximum_probability',
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )

else:

    component_diagnostic = pd.DataFrame(
        columns=component_diagnostic_columns
    )


component_diagnostic.to_csv(
    FOCUS
    / 'top_right_component_diagnostic.csv',
    index=False,
)


# ------------------------------------------------------------
# 11. Save exact crop rows
# ------------------------------------------------------------

crop_output_columns = [
    'page',
    'y',
    'x',
    'component_key',
    'observed_value',
    'paper_value',
    'p_remove_59r2',
    'probability_band',
    'safe_remove',
]


for threshold in THRESHOLDS:

    threshold_label = (
        f'{threshold:.3f}'
    )

    crop_output_columns.extend([
        f'passes_probability_{threshold_label}',
        f'safe_{threshold_label}',
        f'coherence_rejected_{threshold_label}',
    ])


crop_rows[
    crop_output_columns
].to_csv(
    FOCUS
    / 'top_right_region_rows.csv',
    index=False,
)


# ------------------------------------------------------------
# 12. Build diagnostic summary
# ------------------------------------------------------------

selected_by_threshold = {
    f'{threshold:.3f}':
        int(
            crop_rows[
                f'safe_{threshold:.3f}'
            ].sum()
        )

    for threshold
    in THRESHOLDS
}


probability_pass_by_threshold = {
    f'{threshold:.3f}':
        int(
            (
                crop_rows[
                    'p_remove_59r2'
                ]
                >= threshold
            ).sum()
        )

    for threshold
    in THRESHOLDS
}


coherence_rejected_by_threshold = {
    f'{threshold:.3f}':
        int(
            (
                (
                    crop_rows[
                        'p_remove_59r2'
                    ]
                    >= threshold
                )
                & ~crop_rows[
                    f'safe_{threshold:.3f}'
                ]
            ).sum()
        )

    for threshold
    in THRESHOLDS
}


diagnostic_summary = {
    'page':
        FOCUS_PAGE,

    'region_description':
        (
            'Broad upper-right Page258 crop intended to '
            'contain the handwritten SCORE + 27 parts note.'
        ),

    'crop': {
        'x0':
            int(
                crop_x0
            ),

        'y0':
            int(
                crop_y0
            ),

        'x1':
            int(
                crop_x1
            ),

        'y1':
            int(
                crop_y1
            ),

        'width':
            int(
                crop_width
            ),

        'height':
            int(
                crop_height
            ),
    },

    'image_pixels':
        int(
            crop_image_pixels
        ),

    'candidate_rows':
        int(
            len(
                crop_rows
            )
        ),

    'candidate_fraction_of_crop':
        float(
            len(
                crop_rows
            )
            / max(
                crop_image_pixels,
                1,
            )
        ),

    'candidate_components':
        int(
            crop_rows[
                'component_key'
            ].nunique()
        )
        if len(
            crop_rows
        )
        else 0,

    'minimum_p_remove_59r2':
        float(
            crop_rows[
                'p_remove_59r2'
            ].min()
        )
        if len(
            crop_rows
        )
        else None,

    'median_p_remove_59r2':
        float(
            crop_rows[
                'p_remove_59r2'
            ].median()
        )
        if len(
            crop_rows
        )
        else None,

    'mean_p_remove_59r2':
        float(
            crop_rows[
                'p_remove_59r2'
            ].mean()
        )
        if len(
            crop_rows
        )
        else None,

    'maximum_p_remove_59r2':
        float(
            crop_rows[
                'p_remove_59r2'
            ].max()
        )
        if len(
            crop_rows
        )
        else None,

    'selected_by_threshold':
        selected_by_threshold,

    'probability_pass_by_threshold':
        probability_pass_by_threshold,

    'coherence_rejected_by_threshold':
        coherence_rejected_by_threshold,

    'probability_band_counts':
        (
            crop_rows[
                'probability_band'
            ]
            .value_counts()
            .sort_index()
            .to_dict()
        )
        if len(
            crop_rows
        )
        else {},

    'observed_value_summary':
        {
            'minimum':
                int(
                    crop_rows[
                        'observed_value'
                    ].min()
                )
                if len(
                    crop_rows
                )
                else None,

            'median':
                float(
                    crop_rows[
                        'observed_value'
                    ].median()
                )
                if len(
                    crop_rows
                )
                else None,

            'mean':
                float(
                    crop_rows[
                        'observed_value'
                    ].mean()
                )
                if len(
                    crop_rows
                )
                else None,

            'maximum':
                int(
                    crop_rows[
                        'observed_value'
                    ].max()
                )
                if len(
                    crop_rows
                )
                else None,
        },

    'truth_available':
        False,

    'clean_truth_used':
        False,

    'interpretation_rules': {
        'candidate_fraction_near_zero':
            (
                'The note is predominantly outside the '
                'scored candidate population. Threshold '
                'movement cannot remove material that was '
                'never included in the cohort.'
            ),

        'maximum_probability_below_0_950':
            (
                'Candidate rows exist, but their model '
                'probability is below every tested threshold.'
            ),

        'probability_pass_but_not_selected':
            (
                'Rows pass the pixel threshold but are '
                'rejected by page-global coherence.'
            ),

        'selected_rows_but_visible_note_remains':
            (
                'Only part of the visual note is represented '
                'or selected, paper replacement is visually '
                'incomplete, or the crop is broader than the '
                'actual handwritten strokes.'
            ),
    },
}


atomic_json(
    FOCUS
    / 'top_right_region_diagnostic.json',
    diagnostic_summary,
)


# ------------------------------------------------------------
# 13. Console output
# ------------------------------------------------------------

print(
    json.dumps(
        diagnostic_summary,
        indent=2,
    )
)


print(
    '\nProbability-band distribution:'
)


display(
    probability_distribution
)


print(
    '\nTop intersecting components:'
)


display(
    component_diagnostic.head(
        20
    )
)


print(
    '\nCELL 10 STATUS: PASS'
)


print(
    'No clean truth was used or fabricated.'
)

## 11. Integrity verdict, executive summary and curated package

In [ ]:
# ============================================================
# Cell 11 - Final integrity verdict, experiment identity,
#           executive summary and curated upload package
#
# This version reflects the corrected scientific scope:
#
#   - 151-page annotated-only threshold visual audit;
#   - no clean-truth safety claim;
#   - complete threshold ladder for every page;
#   - Page258 top-right probability/candidate diagnosis;
#   - post-outcome diagnostic thresholds only;
#   - no promotion authority.
# ============================================================


# ------------------------------------------------------------
# 1. Required output paths
# ------------------------------------------------------------

required_audit_files = [
    AUDIT
    / 'decision_lock_before_truth.json',

    AUDIT
    / 'clean_truth_availability.json',

    AUDIT
    / 'truth_availability.json',

    AUDIT
    / 'truth_availability.txt',

    AUDIT
    / 'truth_table_identity.json',

    AUDIT
    / 'page_pair_resolution.csv',
]


required_table_files = [
    TABLES
    / 'truth_availability_by_page.csv',

    TABLES
    / 'truth_safety_threshold_summary.csv',

    TABLES
    / 'observable_threshold_transitions.csv',

    TABLES
    / 'all_page_ladder_counts.csv',
]


required_focus_files = [
    FOCUS
    / 'top_right_region_diagnostic.json',

    FOCUS
    / 'top_right_region_rows.csv',

    FOCUS
    / 'top_right_probability_distribution.csv',

    FOCUS
    / 'top_right_component_diagnostic.csv',

    FOCUS
    / 'top_right_score_27_parts_threshold_ladder.png',

    FOCUS
    / 'top_right_candidate_population.png',

    FOCUS
    / 'top_right_probability_overlay.png',
]


# ------------------------------------------------------------
# 2. Verify decision identity has not changed
# ------------------------------------------------------------

require(
    'decision_lock'
    in globals(),
    (
        'decision_lock is unavailable. Run corrected Cell 6.'
    ),
)


require(
    'current_decision_hash'
    in globals(),
    (
        'current_decision_hash is unavailable. '
        'Run corrected Cell 6.'
    ),
)


decision_hash_unchanged = bool(
    current_decision_hash
    == decision_lock[
        'decision_semantic_sha256'
    ]
)


# ------------------------------------------------------------
# 3. Verify complete all-page ladder roster
# ------------------------------------------------------------

ladder_files = sorted(
    LADDERS.glob(
        '*_threshold_ladder.png'
    )
)


ladder_pages = {
    path.name.replace(
        '_threshold_ladder.png',
        '',
    )
    for path
    in ladder_files
}


expected_pages = set(
    cohort[
        'page'
    ].astype(str)
)


missing_ladder_pages = sorted(
    expected_pages
    - ladder_pages
)


unexpected_ladder_pages = sorted(
    ladder_pages
    - expected_pages
)


contact_sheet_files = sorted(
    CONTACTS.glob(
        '*.png'
    )
)


# ------------------------------------------------------------
# 4. Build fail-closed checks
# ------------------------------------------------------------

checks = {
    '151_page_roster':
        bool(
            cohort[
                'page'
            ].nunique()
            == EXPECTED_PAGES
        ),

    'unique_page_y_x':
        bool(
            not cohort.duplicated(
                [
                    'page',
                    'y',
                    'x',
                ]
            ).any()
        ),

    'all_observed_pages_resolved':
        bool(
            len(
                page_resolution
            )
            == EXPECTED_PAGES
        ),

    'page_resolution_roster_exact':
        bool(
            set(
                page_resolution[
                    'page'
                ].astype(str)
            )
            == expected_pages
        ),

    'decision_lock_written':
        bool(
            (
                AUDIT
                / 'decision_lock_before_truth.json'
            ).is_file()
        ),

    'decision_hash_unchanged':
        decision_hash_unchanged,

    'clean_truth_not_fabricated':
        bool(
            truth_table
            is None
        ),

    'truth_scoring_disabled':
        bool(
            not TRUTH_SCORING_AVAILABLE
        ),

    'truth_status_recorded':
        bool(
            (
                AUDIT
                / 'truth_availability.json'
            ).is_file()
        ),

    'frozen_0_995_replay':
        bool(
            (
                AUDIT
                / 'frozen_replay.json'
            ).is_file()
            if (
                AUDIT
                / 'frozen_replay.json'
            ).exists()
            else True
        ),

    'all_page_ladders_written':
        bool(
            len(
                ladder_files
            )
            == EXPECTED_PAGES
        ),

    'all_page_ladder_roster_exact':
        bool(
            not missing_ladder_pages
            and not unexpected_ladder_pages
        ),

    'page258_probability_diagnostic_written':
        bool(
            all(
                path.is_file()
                for path
                in required_focus_files
            )
        ),

    'required_audit_files_written':
        bool(
            all(
                path.is_file()
                for path
                in required_audit_files
            )
        ),

    'required_table_files_written':
        bool(
            all(
                path.is_file()
                for path
                in required_table_files
            )
        ),
}


check_order = [
    '151_page_roster',
    'unique_page_y_x',
    'all_observed_pages_resolved',
    'page_resolution_roster_exact',
    'decision_lock_written',
    'decision_hash_unchanged',
    'clean_truth_not_fabricated',
    'truth_scoring_disabled',
    'truth_status_recorded',
    'frozen_0_995_replay',
    'all_page_ladders_written',
    'all_page_ladder_roster_exact',
    'page258_probability_diagnostic_written',
    'required_audit_files_written',
    'required_table_files_written',
]


first_failure = next(
    (
        check_name
        for check_name
        in check_order
        if not checks[
            check_name
        ]
    ),
    None,
)


verdict = (
    'PASS'
    if first_failure is None
    else 'FAIL'
)


audit_verdict = {
    'verdict':
        verdict,

    'first_failed_check':
        first_failure,

    'checks':
        checks,

    'pages':
        int(
            cohort[
                'page'
            ].nunique()
        ),

    'rows':
        int(
            len(
                cohort
            )
        ),

    'thresholds':
        THRESHOLDS,

    'individual_threshold_ladders':
        int(
            len(
                ladder_files
            )
        ),

    'contact_sheets':
        int(
            len(
                contact_sheet_files
            )
        ),

    'missing_ladder_pages':
        missing_ladder_pages,

    'unexpected_ladder_pages':
        unexpected_ladder_pages,

    'truth_scoring_available':
        bool(
            TRUTH_SCORING_AVAILABLE
        ),

    'clean_truth_fabricated':
        False,

    'scientific_scope':
        (
            '151-page annotated-only observable and visual '
            'diagnostic threshold audit.'
        ),

    'promotion_authority':
        False,
}


atomic_json(
    AUDIT
    / 'notebook64_verdict.json',
    audit_verdict,
)


require(
    verdict
    == 'PASS',
    (
        'Notebook 64 failed its final integrity gate: '
        f'{first_failure}. '
        f'See {AUDIT / "notebook64_verdict.json"}'
    ),
)


# ------------------------------------------------------------
# 5. Read the Page258 diagnostic for the executive summary
# ------------------------------------------------------------

page258_diagnostic = json.loads(
    (
        FOCUS
        / 'top_right_region_diagnostic.json'
    ).read_text()
)


# ------------------------------------------------------------
# 6. Software environment
# ------------------------------------------------------------

environment = {
    'python':
        sys.version,

    'opencv':
        cv2.__version__,

    'numpy':
        np.__version__,

    'pandas':
        pd.__version__,

    'platform':
        platform.platform(),
}


atomic_json(
    RUN
    / 'software_environment.json',
    environment,
)


# ------------------------------------------------------------
# 7. Experiment identity
# ------------------------------------------------------------

experiment_identity = {
    'created_utc':
        datetime.now(
            timezone.utc
        ).isoformat(),

    'version':
        (
            'Notebook64-annotated-only-threshold-'
            'visual-audit-v2'
        ),

    'title':
        (
            '151-Page Annotated-Only Threshold '
            'Visual and Probability Audit'
        ),

    'development_diagnostic_only':
        True,

    'promotion_authority':
        False,

    'clean_truth_available':
        False,

    'clean_truth_fabricated':
        False,

    'truth_side_safety_available':
        False,

    'source_notebook61':
        str(
            NB61
        ),

    'source_parquet':
        str(
            PARQUET_PATH
        ),

    'source_parquet_sha256':
        sha256_file(
            PARQUET_PATH
        ),

    'frozen_59r2':
        str(
            R59R2
        ),

    'policy_id':
        policy.get(
            'policy_id'
        ),

    'compatibility_id':
        policy.get(
            'compatibility_id'
        ),

    'frozen_threshold':
        float(
            FROZEN_THRESHOLD
        ),

    'diagnostic_thresholds':
        THRESHOLDS,

    'connectivity':
        int(
            CONNECTIVITY
        ),

    'minimum_run':
        int(
            MINIMUM_RUN
        ),

    'pages':
        int(
            cohort[
                'page'
            ].nunique()
        ),

    'rows':
        int(
            len(
                cohort
            )
        ),

    'individual_threshold_ladders':
        int(
            len(
                ladder_files
            )
        ),

    'contact_sheets':
        int(
            len(
                contact_sheet_files
            )
        ),

    'decision_lock':
        decision_lock,

    'truth_status':
        truth_status,

    'page258_top_right_diagnostic':
        page258_diagnostic,

    'verdict':
        audit_verdict,

    'software':
        environment,
}


atomic_json(
    RUN
    / 'experiment_identity.json',
    experiment_identity,
)


# ------------------------------------------------------------
# 8. Executive summary
# ------------------------------------------------------------

selected_lines = []


for threshold in THRESHOLDS:

    selected_pixels = int(
        cohort[
            f'safe_{threshold:.3f}'
        ].sum()
    )

    selected_pages = int(
        (
            cohort
            .groupby(
                'page'
            )[
                f'safe_{threshold:.3f}'
            ]
            .sum()
            > 0
        ).sum()
    )


    selected_lines.append(
        (
            f'{threshold:.3f}: '
            f'{selected_pixels:,} selected pixels '
            f'across {selected_pages} pages'
        )
    )


page258_selected = (
    page258_diagnostic.get(
        'selected_by_threshold',
        {}
    )
)


page258_pass = (
    page258_diagnostic.get(
        'probability_pass_by_threshold',
        {}
    )
)


page258_coherence = (
    page258_diagnostic.get(
        'coherence_rejected_by_threshold',
        {}
    )
)


executive_lines = [
    (
        'NOTEBOOK 64 - 151-PAGE ANNOTATED-ONLY '
        'THRESHOLD VISUAL AUDIT'
    ),

    '',

    f'VERDICT: {verdict}',

    (
        'Pages: '
        f'{cohort.page.nunique()}'
    ),

    (
        'Scored rows: '
        f'{len(cohort):,}'
    ),

    (
        'Individual page threshold ladders: '
        f'{len(ladder_files)}'
    ),

    (
        'Contact sheets: '
        f'{len(contact_sheet_files)}'
    ),

    '',

    'DIAGNOSTIC THRESHOLD COVERAGE',

    *selected_lines,

    '',

    'PAGE258 TOP-RIGHT DIAGNOSTIC',

    (
        'Crop coordinates: '
        f'{page258_diagnostic.get("crop")}'
    ),

    (
        'Crop image pixels: '
        f'{page258_diagnostic.get("image_pixels", 0):,}'
    ),

    (
        'Scored candidate rows in crop: '
        f'{page258_diagnostic.get("candidate_rows", 0):,}'
    ),

    (
        'Candidate fraction of crop: '
        f'{page258_diagnostic.get("candidate_fraction_of_crop", 0):.6f}'
    ),

    (
        'Minimum / median / maximum p_remove_59r2: '
        f'{page258_diagnostic.get("minimum_p_remove_59r2")} / '
        f'{page258_diagnostic.get("median_p_remove_59r2")} / '
        f'{page258_diagnostic.get("maximum_p_remove_59r2")}'
    ),

    (
        'Selected crop rows by threshold: '
        f'{page258_selected}'
    ),

    (
        'Probability-passing crop rows by threshold: '
        f'{page258_pass}'
    ),

    (
        'Coherence-rejected crop rows by threshold: '
        f'{page258_coherence}'
    ),

    '',

    'TRUTH STATUS',

    (
        'Clean truth is unavailable for the complete '
        '151-page annotated-only cohort.'
    ),

    (
        'No PRINT / ANNOTATION_NONPRINT / UNCERTAIN '
        'clean-derived labels were fabricated.'
    ),

    (
        'This notebook measures threshold response, '
        'coverage, concentration, probability localisation '
        'and whole-page visual behaviour.'
    ),

    (
        'This notebook does not establish false-removal '
        'safety against genuine clean printed ink.'
    ),

    '',

    'INTERPRETATION BOUNDARY',

    (
        'All thresholds other than frozen 0.995 are '
        'retrospective diagnostic operating points.'
    ),

    (
        'No threshold examined here is a frozen challenger, '
        'independent validation result or production '
        'authorisation.'
    ),
]


(
    RUN
    / 'executive_summary.txt'
).write_text(
    '\n'.join(
        executive_lines
    )
    + '\n'
)


print(
    '\n'.join(
        executive_lines
    )
)


# ------------------------------------------------------------
# 9. Curated upload package
#
# The package includes:
#
#   - summary and identity records;
#   - audit status;
#   - compact diagnostic tables;
#   - Page258 top-right diagnostics;
#   - all 151 individual threshold ladders;
#   - all batched contact sheets.
#
# It does not include the large source parquet.
# ------------------------------------------------------------

curated_files = [
    RUN
    / 'executive_summary.txt',

    RUN
    / 'experiment_identity.json',

    RUN
    / 'software_environment.json',

    AUDIT
    / 'notebook64_verdict.json',

    AUDIT
    / 'decision_lock_before_truth.json',

    AUDIT
    / 'clean_truth_availability.json',

    AUDIT
    / 'truth_availability.json',

    AUDIT
    / 'truth_availability.txt',

    AUDIT
    / 'truth_table_identity.json',

    AUDIT
    / 'page_pair_resolution.csv',

    TABLES
    / 'truth_availability_by_page.csv',

    TABLES
    / 'truth_safety_threshold_summary.csv',

    TABLES
    / 'observable_threshold_transitions.csv',

    TABLES
    / 'all_page_ladder_counts.csv',

    FOCUS
    / 'top_right_region_diagnostic.json',

    FOCUS
    / 'top_right_region_rows.csv',

    FOCUS
    / 'top_right_probability_distribution.csv',

    FOCUS
    / 'top_right_component_diagnostic.csv',

    FOCUS
    / 'top_right_score_27_parts_threshold_ladder.png',

    FOCUS
    / 'top_right_candidate_population.png',

    FOCUS
    / 'top_right_probability_overlay.png',
]


# Include optional Notebook 63-style tables if present.

optional_files = [
    TABLES
    / 'threshold_envelope_summary.csv',

    TABLES
    / 'decision_oriented_threshold_summary.csv',

    TABLES
    / 'page_rank_stability.csv',

    TABLES
    / 'threshold_results_by_page.csv',

    TABLES
    / 'threshold_results_by_source_group.csv',

    TABLES
    / 'threshold_results_by_component.csv',

    TABLES
    / 'visual_review_queue.csv',

    RUN
    / 'threshold_envelope_dashboard.png',

    RUN
    / 'page_concentration_curves.png',
]


curated_files.extend([
    path
    for path
    in optional_files
    if path.is_file()
])


curated_files.extend(
    ladder_files
)


curated_files.extend(
    contact_sheet_files
)


# Ensure each path occurs only once.

unique_curated_files = []


seen_curated_paths = set()


for path in curated_files:

    path = Path(
        path
    )


    key = str(
        path.resolve()
    )


    if key in seen_curated_paths:

        continue


    seen_curated_paths.add(
        key
    )


    unique_curated_files.append(
        path
    )


curated_files = (
    unique_curated_files
)


missing_curated_files = [
    str(
        path
    )
    for path
    in curated_files
    if not path.is_file()
]


require(
    not missing_curated_files,
    (
        'Required curated artefacts are missing: '
        f'{missing_curated_files}'
    ),
)


# ------------------------------------------------------------
# 10. Copy curated artefacts with hash verification
# ------------------------------------------------------------

copy_manifest_rows = []


for source_path in curated_files:

    relative_path = (
        source_path.relative_to(
            RUN
        )
    )


    destination_path = (
        UPLOAD
        / relative_path
    )


    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    shutil.copy2(
        source_path,
        destination_path,
    )


    source_hash = (
        sha256_file(
            source_path
        )
    )


    destination_hash = (
        sha256_file(
            destination_path
        )
    )


    require(
        source_hash
        == destination_hash,
        (
            'Upload-copy hash mismatch: '
            f'{relative_path}'
        ),
    )


    copy_manifest_rows.append({
        'relative_path':
            str(
                relative_path
            ),

        'bytes':
            int(
                destination_path.stat().st_size
            ),

        'sha256':
            destination_hash,
    })


# ------------------------------------------------------------
# 11. README and file list
# ------------------------------------------------------------

(
    UPLOAD
    / 'README.txt'
).write_text(
    (
        'Notebook 64 annotated-only threshold visual audit.\n'
        '\n'
        'This package contains observable and visual evidence '
        'for the 151 reviewed annotated-only pages.\n'
        '\n'
        'Clean truth is unavailable for the complete cohort. '
        'No clean-derived truth labels were fabricated.\n'
        '\n'
        'Alternative thresholds are retrospective diagnostic '
        'operating points only. They are not frozen policies, '
        'independent validation or production authorisation.\n'
        '\n'
        'The package includes one seven-panel threshold ladder '
        'for every reviewed page and a dedicated Page258 '
        'top-right candidate/probability diagnostic.\n'
        '\n'
        'The large source parquet is not included.\n'
    )
)


# The list excludes upload_file_list.txt itself so the contents
# remain stable after the file is written.

package_files_before_list = sorted(
    str(
        path.relative_to(
            UPLOAD
        )
    )
    for path
    in UPLOAD.rglob(
        '*'
    )
    if (
        path.is_file()
        and path.name
        != 'upload_file_list.txt'
        and path.name
        != 'upload_manifest.csv'
    )
)


(
    UPLOAD
    / 'upload_file_list.txt'
).write_text(
    '\n'.join(
        package_files_before_list
    )
    + '\n'
)


# ------------------------------------------------------------
# 12. Final package manifest
#
# Avoid the earlier duplicate-manifest-row problem by scanning
# the completed upload directory once and enforcing uniqueness.
# ------------------------------------------------------------

upload_manifest_rows = []


for path in sorted(
    UPLOAD.rglob(
        '*'
    )
):

    if (
        not path.is_file()
        or path.name
        == 'upload_manifest.csv'
    ):

        continue


    upload_manifest_rows.append({
        'relative_path':
            str(
                path.relative_to(
                    UPLOAD
                )
            ),

        'bytes':
            int(
                path.stat().st_size
            ),

        'sha256':
            sha256_file(
                path
            ),
    })


upload_manifest = (
    pd.DataFrame(
        upload_manifest_rows
    )
    .drop_duplicates(
        subset=[
            'relative_path'
        ],
        keep='first',
    )
    .sort_values(
        'relative_path'
    )
    .reset_index(
        drop=True
    )
)


require(
    not upload_manifest[
        'relative_path'
    ].duplicated().any(),
    (
        'Duplicate relative paths remain in upload manifest.'
    ),
)


upload_manifest.to_csv(
    UPLOAD
    / 'upload_manifest.csv',
    index=False,
)


# ------------------------------------------------------------
# 13. Create and verify ZIP
# ------------------------------------------------------------

zip_path = (
    RUN
    / (
        'Notebook64_Annotated_Only_'
        'Threshold_Visual_Audit_upload_bundle.zip'
    )
)


with zipfile.ZipFile(
    zip_path,
    'w',
    zipfile.ZIP_DEFLATED,
) as archive:

    for path in sorted(
        UPLOAD.rglob(
            '*'
        )
    ):

        if path.is_file():

            archive.write(
                path,
                path.relative_to(
                    UPLOAD
                ),
            )


with zipfile.ZipFile(
    zip_path,
    'r',
) as archive:

    corrupt_member = (
        archive.testzip()
    )


require(
    corrupt_member is None,
    (
        'ZIP integrity test failed at: '
        f'{corrupt_member}'
    ),
)


# ------------------------------------------------------------
# 14. Package identity
# ------------------------------------------------------------

package_identity = {
    'created_utc':
        datetime.now(
            timezone.utc
        ).isoformat(),

    'zip_path':
        str(
            zip_path
        ),

    'zip_bytes':
        int(
            zip_path.stat().st_size
        ),

    'zip_sha256':
        sha256_file(
            zip_path
        ),

    'upload_files':
        int(
            len(
                upload_manifest
            )
        ),

    'individual_threshold_ladders':
        int(
            len(
                ladder_files
            )
        ),

    'contact_sheets':
        int(
            len(
                contact_sheet_files
            )
        ),

    'source_parquet_included':
        False,

    'clean_truth_included':
        False,

    'promotion_authority':
        False,
}


atomic_json(
    RUN
    / 'upload_package_identity.json',
    package_identity,
)


# ------------------------------------------------------------
# 15. Final console output
# ------------------------------------------------------------

print(
    '\nCELL 11 STATUS: PASS'
)


print(
    'Notebook 64 verdict:',
    verdict,
)


print(
    'Individual threshold ladders:',
    len(
        ladder_files
    ),
)


print(
    'Contact sheets:',
    len(
        contact_sheet_files
    ),
)


print(
    'Upload files:',
    len(
        upload_manifest
    ),
)


print(
    'Upload ZIP:',
    zip_path,
)


print(
    'Upload ZIP SHA-256:',
    package_identity[
        'zip_sha256'
    ],
)


print(
    '\nClean truth was not used or fabricated.'
)


print(
    'Truth-side safety remains unavailable for this cohort.'
)